# gas_trend 02 — Does a trend model warn earlier than the chain?
Pass criteria were fixed in `FICHE.md` before running: numpy GRU = PyTorch (≤ 1e-5, unit test), no node shared (unit test), **false flags per hour on clean air ≤ the chain's**, lead time reported with a bootstrap CI (no pass level; if the CI includes 0 → "no earlier than the chain"), < 1 ms per update.
Models: `chain_depth` (threshold on the chain's depth: the chain itself with a lower threshold), `slope_rule` (extrapolate depth 8 scans ahead), `logistic` (window features), `gru` (hidden 12, 769 parameters).

In [1]:
import sys, pathlib, time
import numpy as np
ROOT = pathlib.Path.cwd().resolve().parents[1]
sys.path[:0] = [str(ROOT / "python/src"), str(ROOT.parent / "common/python/src"), str(ROOT.parents[1] / "box/gas_chain/python/src")]
from gas_chain import data_mendeley as D
from gas_trend import experiments as E
data = D.load_all()

In [2]:
runs = {}
for factor in (1.0, 0.5, 0.25):
    runs[factor] = E.run_factor(factor, data, gru_seeds=3, epochs=40)
    print("factor", factor, "done")

factor 1.0 done


factor 0.5 done


factor 0.25 done


## Lead time and false alarms (median over episodes; missed = lead 0; CI = bootstrap over recordings)

In [3]:
print(f"{'factor':>6s} {'model':12s} {'median lead s':>13s} {'95% CI':>16s} {'episodes':>8s} {'missed':>7s} {'FA/h nodes 1..5':>30s}   chain FA/h")
for factor, run in runs.items():
    for m in E.MODELS:
        med, lo, hi, n = E.lead_summary(run, m)
        missed = sum(r["out"][m]["missed"] for r in run["res"].values())
        fa = [round(r["out"][m]["fa_per_hour"], 1) for r in run["res"].values()]
        cfa = [round(float(r["out"][m]["chain_fa_per_hour"]), 1) for r in run["res"].values()]
        print(f"{factor:6.2f} {m:12s} {med:13.0f} [{lo:6.0f},{hi:6.0f}] {n:8d} {missed:7d} {str(fa):>30s}   {cfa if m == 'gru' else ''}")

factor model        median lead s           95% CI episodes  missed                FA/h nodes 1..5   chain FA/h
  1.00 chain_depth              0 [     0,   776]       38      25   [118.5, 0.0, 14.1, 0.0, 0.0]   
  1.00 slope_rule               0 [     0,   776]       38      25   [117.8, 0.0, 14.1, 0.0, 0.0]   


  1.00 logistic                 0 [     0,   978]       38      21    [65.0, 0.0, 7.4, 21.1, 0.0]   


  1.00 gru                      0 [     0,     0]       38      31     [0.0, 0.0, 17.6, 2.9, 0.0]   [0.0, 0.0, 9.0, 0.0, 0.0]
  0.50 chain_depth              0 [     0,   424]       38      27   [118.5, 0.0, 10.7, 0.0, 0.0]   


  0.50 slope_rule               0 [     0,   834]       38      24    [117.5, 0.0, 9.5, 0.0, 0.0]   


  0.50 logistic                99 [     0,  1024]       38      19   [93.8, 0.0, 0.0, 136.7, 0.0]   
  0.50 gru                      0 [     0,     0]       38      28   [0.0, 0.0, 32.5, 101.2, 0.0]   [0.0, 0.0, 4.0, 0.0, 0.0]
  0.25 chain_depth              0 [     0,   857]       38      25   [118.5, 0.0, 14.1, 0.0, 0.0]   


  0.25 slope_rule             162 [     0,   984]       38      19   [117.5, 0.0, 13.7, 0.0, 0.0]   
  0.25 logistic               428 [     0,  1232]       38      18   [91.6, 0.0, 0.0, 133.5, 0.0]   
  0.25 gru                    520 [     0,  1041]       38      18  [57.7, 0.0, 53.9, 128.3, 0.0]   [0.0, 0.0, 3.6, 0.0, 0.0]


## Window-level ranking (AUROC / AP of 'onset within 8 scans', per test node, factor 1.0)

In [4]:
for m in E.MODELS:
    print(f"{m:12s} AUROC", [round(r["out"][m]["auc"], 2) for r in runs[1.0]["res"].values()], " AP", [round(r["out"][m]["ap"], 2) for r in runs[1.0]["res"].values()])
print("positive rate:", round(float(runs[1.0]["meta"]["y"].mean()), 3))

chain_depth  AUROC [0.49, 0.58, 0.71, 0.5, 0.5]  AP [0.08, 0.46, 0.28, 0.05, 0.15]
slope_rule   AUROC [0.45, 0.55, 0.54, 0.5, 0.5]  AP [0.07, 0.4, 0.13, 0.05, 0.15]
logistic     AUROC [0.71, 0.63, 0.25, 0.8, 0.6]  AP [0.14, 0.5, 0.05, 0.14, 0.23]
gru          AUROC [0.9, 0.84, 0.62, 0.36, 0.17]  AP [0.49, 0.64, 0.17, 0.05, 0.1]
positive rate: 0.098


## How to read the result
1. **Criterion 'false flags ≤ chain's'**: look at the FA/h columns against the chain's. A model that exceeds the chain's rate on a test node *fails* the criterion on that node, whatever its lead time — a lead time bought with false alarms is worthless.
2. **Lead time**: if the CI includes 0, the conclusion is "no earlier than the chain".
3. Small sample: 38 episodes from 20 recordings of 2 burns; intervals are wide and the dilution factors are not independent experiments.